# 03 - Feature Engineering

## SmartPlay AI Recommendation System

This notebook performs feature engineering on the preprocessed Grade 1 student learning data.

The objective is to transform the cleaned datasets into meaningful student-level features that can be used for:

1. Personalized learning-path recommendation
2. Learning-outcome prediction

| # | Column | Role |
|---:|---|---|
| 1 | `Student_ID` | Identifier — exclude from model |
| 2–15 | Performance features | Input features |
| 16–21 | Teacher/learner categorical features | Input features |
| 22–29 | Engineered historical/behavior features | Input features |

In [82]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import os

In [83]:
PROCESSED_PATH = "../data/processed"
print(os.listdir(PROCESSED_PATH))

['model_evaluation_metrics.csv', 'recommendation_test_results.csv', 'final_personalized_learning_paths.csv', 'assessment_clean.csv', 'model_comparison.csv', '.DS_Store', 'student_learning_path_validation.csv', 'cv_prediction_results.csv', 'ai_learning_clean.csv', '.gitkeep', 'validated_personalized_learning_paths.csv', 'model_accuracy_comparison.png', 'activity_clean.csv', 'model_ranking.csv', 'model_f1_comparison.png', 'curriculum_clean.csv', 'teacher_clean.csv', 'student_clean.csv', 'student_learning_path_summary.csv', 'historical_clean.csv', 'student_features.csv']


In [84]:
# ============================================================
# 03 - FEATURE ENGINEERING
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [85]:
# ============================================================
# SET PROCESSED DATA PATH
# ============================================================

PROCESSED_PATH = "../data/processed"

print("Processed data path:", PROCESSED_PATH)

Processed data path: ../data/processed


In [86]:
# ============================================================
# CHECK PROCESSED DATASETS
# ============================================================

print("Files available in processed directory:")

for file in sorted(os.listdir(PROCESSED_PATH)):
    if file.endswith(".csv"):
        print("✓", file)

Files available in processed directory:
✓ activity_clean.csv
✓ ai_learning_clean.csv
✓ assessment_clean.csv
✓ curriculum_clean.csv
✓ cv_prediction_results.csv
✓ final_personalized_learning_paths.csv
✓ historical_clean.csv
✓ model_comparison.csv
✓ model_evaluation_metrics.csv
✓ model_ranking.csv
✓ recommendation_test_results.csv
✓ student_clean.csv
✓ student_features.csv
✓ student_learning_path_summary.csv
✓ student_learning_path_validation.csv
✓ teacher_clean.csv
✓ validated_personalized_learning_paths.csv


In [87]:
# ============================================================
# LOAD PREPROCESSED DATASETS
# ============================================================

student_clean = pd.read_csv(
    f"{PROCESSED_PATH}/student_clean.csv"
)

curriculum_clean = pd.read_csv(
    f"{PROCESSED_PATH}/curriculum_clean.csv"
)

activity_clean = pd.read_csv(
    f"{PROCESSED_PATH}/activity_clean.csv"
)

assessment_clean = pd.read_csv(
    f"{PROCESSED_PATH}/assessment_clean.csv"
)

teacher_clean = pd.read_csv(
    f"{PROCESSED_PATH}/teacher_clean.csv"
)

historical_clean = pd.read_csv(
    f"{PROCESSED_PATH}/historical_clean.csv"
)

ai_learning_clean = pd.read_csv(
    f"{PROCESSED_PATH}/ai_learning_clean.csv"
)

print("All preprocessed datasets loaded successfully.")

All preprocessed datasets loaded successfully.


In [88]:
# ============================================================
# CHECK DATASET SHAPES
# ============================================================

datasets = {
    "Student": student_clean,
    "Curriculum": curriculum_clean,
    "Activity": activity_clean,
    "Assessment": assessment_clean,
    "Teacher Observation": teacher_clean,
    "Historical Learning Path": historical_clean,
    "AI Learning Path": ai_learning_clean
}

print("=" * 70)
print("DATASET SHAPES")
print("=" * 70)

for name, data in datasets.items():
    print(
        f"{name:25} "
        f"Rows={data.shape[0]:3} "
        f"Columns={data.shape[1]:2}"
    )

DATASET SHAPES
Student                   Rows=100 Columns= 8
Curriculum                Rows= 20 Columns= 9
Activity                  Rows= 60 Columns=12
Assessment                Rows= 30 Columns=15
Teacher Observation       Rows=100 Columns= 9
Historical Learning Path  Rows= 60 Columns= 6
AI Learning Path          Rows= 11 Columns=14


In [89]:
# ============================================================
# INSPECT STUDENT DATA
# ============================================================

print("Student dataset:")
display(student_clean.head())

print("\nStudent columns:")
print(student_clean.columns.tolist())

Student dataset:


,Student_ID,Student_Name,Age,Gender,School Name,District,Grade,Learning_Environment
0,ST001,Ahamed Rizwan,6,Male,AK/Iqrah VIdyalaya - Addalaichenai - Home,Ampara,Grade 1,Classroom + Digital Learning
1,ST002,Aisha Fathima,7,Female,AK/Iqrah VIdyalaya - Addalaichenai - Home,Ampara,Grade 1,Classroom Only
2,ST003,Ayaan Nazeer,7,Male,AK/Iqrah VIdyalaya - Addalaichenai - Home,Ampara,Grade 1,Home + Classroom
3,ST004,Fathima Zahra,6,Female,AK/Iqrah VIdyalaya - Addalaichenai - Home,Ampara,Grade 1,Blended Learning
4,ST005,Ibrahim Faiz,7,Male,AK/Iqrah VIdyalaya - Addalaichenai - Home,Ampara,Grade 1,Activity-Based Learning



Student columns:
['Student_ID', 'Student_Name', 'Age', 'Gender', 'School Name', 'District', 'Grade', 'Learning_Environment']


In [90]:
# ============================================================
# INSPECT ACTIVITY DATA
# ============================================================

print("Activity dataset:")
display(activity_clean.head())

print("\nActivity columns:")
print(activity_clean.columns.tolist())

Activity dataset:


,Activity_ID,Student_ID,Lesson_ID,Activity_Type,Difficulty,Date,Completion_Status,Attempts,Time_Taken_Min,Score,Errors,Teacher_Evaluation
0,ACT001,ST001,LES001,Number Game,Easy,2026-01-10,Completed,1,8,95,None recorded,Excellent
1,ACT002,ST001,LES003,Phonics Exercise,Medium,2026-01-12,Completed,3,15,55,Letter sound confusion,Needs support
2,ACT003,ST002,LES001,Worksheet,Easy,2026-01-15,Completed,1,10,85,Minor mistakes,Good
3,ACT004,ST002,LES006,Pencil Tracing,Easy,2026-01-16,Completed,2,12,70,Uneven lines,Improving
4,ACT005,ST003,LES009,Creative Drawing,Medium,2026-01-17,Completed,1,18,92,None recorded,Excellent



Activity columns:
['Activity_ID', 'Student_ID', 'Lesson_ID', 'Activity_Type', 'Difficulty', 'Date', 'Completion_Status', 'Attempts', 'Time_Taken_Min', 'Score', 'Errors', 'Teacher_Evaluation']


In [91]:
# ============================================================
# INSPECT ASSESSMENT DATA
# ============================================================

print("Assessment dataset:")
display(assessment_clean.head())

print("\nAssessment columns:")
print(assessment_clean.columns.tolist())

Assessment dataset:


,Assessment_ID,Student_ID,Subject,Assessment_Type,Pre_Test,Post_Test,Unit_Test,Monthly_Test,Term_Result,Competency_Score,Motor_Skills_Score,Creativity_Score,Interactive_Engagement,Learning_Progress,Recommended_Action
0,ASS001,ST001,Mathematics,Number Recognition,60,90,88,85,87,Number Recognition: 90%,75,70,90,30,Number puzzle activity
1,ASS002,ST001,English,Phonics,40,65,60,62,63,Phonics: 65%,70,75,85,25,Interactive phonics games
2,ASS003,ST002,Mathematics,Counting Objects,45,70,68,72,70,Counting: 70%,80,65,75,25,Counting with manipulatives
3,ASS004,ST002,Fine Motor Skills,Pencil Control,50,60,62,64,63,Pencil Control: 63%,60,65,70,10,Guided tracing exercises
4,ASS005,ST003,Creativity,Drawing Shapes,55,85,82,80,83,Shape Creation: 85%,80,90,88,30,Open-ended drawing



Assessment columns:
['Assessment_ID', 'Student_ID', 'Subject', 'Assessment_Type', 'Pre_Test', 'Post_Test', 'Unit_Test', 'Monthly_Test', 'Term_Result', 'Competency_Score', 'Motor_Skills_Score', 'Creativity_Score', 'Interactive_Engagement', 'Learning_Progress', 'Recommended_Action']


In [92]:
# ============================================================
# INSPECT TEACHER OBSERVATION DATA
# ============================================================

print("Teacher observation dataset:")
display(teacher_clean.head())

print("\nTeacher observation columns:")
print(teacher_clean.columns.tolist())

Teacher observation dataset:


,Student_ID,Participation,Attention_Level,Confidence,Collaboration,Learning_Difficulty,Learning_Style,Teacher_Comments,Recommendation
0,ST001,High,Medium,High,Good,Reading difficulty,Visual learner,Understands concepts quickly,More reading activities
1,ST002,Low,Low,Medium,Needs improvement,Slow writing,Practical learner,Requires guidance,Individual support
2,ST003,High,High,High,Good,None identified,Visual learner,Enjoys picture-based tasks,Advanced visual activities
3,ST004,Medium,Medium,Medium,Good,Number recognition,Practical learner,Needs repeated examples,Counting games
4,ST005,High,High,High,Excellent,None identified,Creative learner,Produces imaginative drawings,Open-ended art activities



Teacher observation columns:
['Student_ID', 'Participation', 'Attention_Level', 'Confidence', 'Collaboration', 'Learning_Difficulty', 'Learning_Style', 'Teacher_Comments', 'Recommendation']


In [93]:
# ============================================================
# INSPECT HISTORICAL LEARNING PATH
# ============================================================

print("Historical learning-path dataset:")
display(historical_clean.head())

print("\nHistorical learning-path columns:")
print(historical_clean.columns.tolist())

Historical learning-path dataset:


,Student_ID,Lesson_ID,Sequence_No,Status,Activity_Sequence,Outcome
0,ST001,LES001,1,Completed,Counting → Numbers,Successful
1,ST001,LES003,2,Completed,Letters → Sounds → Words,Needs Practice
2,ST001,LES004,3,Not Started,Reading Sentences,Waiting
3,ST001,LES005,4,In Progress,Number Matching → Number Puzzle,Improving
4,ST002,LES001,1,Completed,Counting Objects → Number Recognition,Successful



Historical learning-path columns:
['Student_ID', 'Lesson_ID', 'Sequence_No', 'Status', 'Activity_Sequence', 'Outcome']


In [94]:
# ============================================================
# STEP 11 - STUDENT ACTIVITY FEATURES
# ============================================================

activity_features = activity_clean.groupby("Student_ID").agg(
    Average_Score=("Score", "mean"),
    Average_Attempts=("Attempts", "mean"),
    Average_Time_Taken=("Time_Taken_Min", "mean"),
    Total_Activities=("Activity_ID", "count")
).reset_index()

print("Student activity features:")
display(activity_features.head())

print("\nShape:", activity_features.shape)

Student activity features:


,Student_ID,Average_Score,Average_Attempts,Average_Time_Taken,Total_Activities
0,ST001,78.00,1.75,11.25,4
1,ST002,78.75,1.50,11.25,4
2,ST003,92.25,1.00,16.00,4
3,ST004,68.75,2.00,11.75,4
4,ST005,95.25,1.00,16.00,4



Shape: (15, 5)


In [95]:
# ============================================================
# STEP 12 - COMPLETION RATE
# ============================================================

completion_rate = (
    activity_clean
    .groupby("Student_ID")["Completion_Status"]
    .apply(lambda x: (x == "Completed").mean() * 100)
    .reset_index(name="Completion_Rate")
)

print("Completion rate:")
display(completion_rate.head())

Completion rate:


,Student_ID,Completion_Rate
0,ST001,100.0
1,ST002,100.0
2,ST003,100.0
3,ST004,75.0
4,ST005,100.0


In [96]:
# ============================================================
# STEP 13 - COMBINE ACTIVITY FEATURES
# ============================================================

activity_features = activity_features.merge(
    completion_rate,
    on="Student_ID",
    how="left"
)

print("Combined activity features:")
display(activity_features.head())

print("\nShape:", activity_features.shape)

Combined activity features:


,Student_ID,Average_Score,Average_Attempts,Average_Time_Taken,Total_Activities,Completion_Rate
0,ST001,78.00,1.75,11.25,4,100.0
1,ST002,78.75,1.50,11.25,4,100.0
2,ST003,92.25,1.00,16.00,4,100.0
3,ST004,68.75,2.00,11.75,4,75.0
4,ST005,95.25,1.00,16.00,4,100.0



Shape: (15, 6)


In [97]:
# ============================================================
# STEP 14 - ASSESSMENT FEATURES
# ============================================================

assessment_numeric_columns = [
    "Pre_Test",
    "Post_Test",
    "Unit_Test",
    "Monthly_Test",
    "Term_Result",
    "Motor_Skills_Score",
    "Creativity_Score",
    "Interactive_Engagement"
]

assessment_features = (
    assessment_clean
    .groupby("Student_ID")[assessment_numeric_columns]
    .mean()
    .reset_index()
)

print("Assessment features:")
display(assessment_features.head())

print("\nShape:", assessment_features.shape)

Assessment features:


,Student_ID,Pre_Test,Post_Test,Unit_Test,Monthly_Test,Term_Result,Motor_Skills_Score,Creativity_Score,Interactive_Engagement
0,ST001,50.0,77.5,74.0,73.5,75.0,72.5,72.5,87.5
1,ST002,47.5,65.0,65.0,68.0,66.5,70.0,65.0,72.5
2,ST003,60.0,82.5,80.0,81.0,81.5,77.5,85.0,86.5
3,ST004,37.5,57.5,58.0,61.0,59.5,65.0,65.0,70.0
4,ST005,60.0,85.0,85.0,88.5,86.5,80.0,87.5,91.0



Shape: (15, 9)


In [98]:
# ============================================================
# STEP 15 - ASSESSMENT IMPROVEMENT
# ============================================================

assessment_clean["Test_Improvement"] = (
    assessment_clean["Post_Test"]
    - assessment_clean["Pre_Test"]
)

improvement_features = (
    assessment_clean
    .groupby("Student_ID")["Test_Improvement"]
    .mean()
    .reset_index()
)

assessment_features = assessment_features.merge(
    improvement_features,
    on="Student_ID",
    how="left"
)

print("Assessment features with improvement:")
display(assessment_features.head())

Assessment features with improvement:


,Student_ID,Pre_Test,Post_Test,Unit_Test,Monthly_Test,Term_Result,Motor_Skills_Score,Creativity_Score,Interactive_Engagement,Test_Improvement
0,ST001,50.0,77.5,74.0,73.5,75.0,72.5,72.5,87.5,27.5
1,ST002,47.5,65.0,65.0,68.0,66.5,70.0,65.0,72.5,17.5
2,ST003,60.0,82.5,80.0,81.0,81.5,77.5,85.0,86.5,22.5
3,ST004,37.5,57.5,58.0,61.0,59.5,65.0,65.0,70.0,20.0
4,ST005,60.0,85.0,85.0,88.5,86.5,80.0,87.5,91.0,25.0


In [99]:
# ============================================================
# STEP 16 - ENCODE TEACHER OBSERVATION FEATURES
# ============================================================

level_mapping = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

teacher_features = teacher_clean[
    [
        "Student_ID",
        "Participation",
        "Attention_Level",
        "Confidence",
        "Collaboration",
        "Learning_Difficulty",
        "Learning_Style"
    ]
].copy()

teacher_features["Participation_Score"] = (
    teacher_features["Participation"].map(level_mapping)
)

teacher_features["Attention_Score"] = (
    teacher_features["Attention_Level"].map(level_mapping)
)

teacher_features["Confidence_Score"] = (
    teacher_features["Confidence"].map(level_mapping)
)

print("Teacher features:")
display(teacher_features.head())

Teacher features:


,Student_ID,Participation,Attention_Level,Confidence,Collaboration,Learning_Difficulty,Learning_Style,Participation_Score,Attention_Score,Confidence_Score
0,ST001,High,Medium,High,Good,Reading difficulty,Visual learner,3,2,3
1,ST002,Low,Low,Medium,Needs improvement,Slow writing,Practical learner,1,1,2
2,ST003,High,High,High,Good,None identified,Visual learner,3,3,3
3,ST004,Medium,Medium,Medium,Good,Number recognition,Practical learner,2,2,2
4,ST005,High,High,High,Excellent,None identified,Creative learner,3,3,3


In [100]:
# ============================================================
# STEP 17 - COLLABORATION SCORE
# ============================================================

collaboration_mapping = {
    "Fair": 1,
    "Needs improvement": 2,
    "Good": 3,
    "Excellent": 4
}

teacher_features["Collaboration_Score"] = (
    teacher_features["Collaboration"]
    .map(collaboration_mapping)
)

print(
    teacher_features[
        [
            "Student_ID",
            "Participation_Score",
            "Attention_Score",
            "Confidence_Score",
            "Collaboration_Score"
        ]
    ].head()
)

  Student_ID  Participation_Score  Attention_Score  Confidence_Score  \
0      ST001                    3                2                 3   
1      ST002                    1                1                 2   
2      ST003                    3                3                 3   
3      ST004                    2                2                 2   
4      ST005                    3                3                 3   

   Collaboration_Score  
0                    3  
1                    2  
2                    3  
3                    3  
4                    4  


In [101]:
# ============================================================
# STEP 18 - HISTORICAL LEARNING PATH FEATURES
# ============================================================

historical_features = (
    historical_clean
    .groupby("Student_ID")
    .agg(
        Historical_Lessons=("Lesson_ID", "count"),
        Average_Sequence_No=("Sequence_No", "mean"),
        Completed_Historical_Lessons=(
            "Status",
            lambda x: (x == "Completed").sum()
        )
    )
    .reset_index()
)

historical_features["Historical_Completion_Rate"] = (
    historical_features["Completed_Historical_Lessons"]
    / historical_features["Historical_Lessons"]
    * 100
)

print("Historical learning-path features:")
display(historical_features.head())

Historical learning-path features:


,Student_ID,Historical_Lessons,Average_Sequence_No,Completed_Historical_Lessons,Historical_Completion_Rate
0,ST001,4,2.5,2,50.0
1,ST002,4,2.5,2,50.0
2,ST003,4,2.5,2,50.0
3,ST004,4,2.5,1,25.0
4,ST005,4,2.5,3,75.0


In [102]:
# ============================================================
# STEP 19 - MERGE ALL STUDENT FEATURES
# ============================================================

student_features = activity_features.merge(
    assessment_features,
    on="Student_ID",
    how="left"
)

student_features = student_features.merge(
    teacher_features,
    on="Student_ID",
    how="left"
)

student_features = student_features.merge(
    historical_features,
    on="Student_ID",
    how="left"
)

print("=" * 70)
print("FINAL STUDENT FEATURE DATASET")
print("=" * 70)

print("Shape:", student_features.shape)

display(student_features.head())

FINAL STUDENT FEATURE DATASET
Shape: (15, 29)


,Student_ID,Average_Score,Average_Attempts,Average_Time_Taken,Total_Activities,Completion_Rate,Pre_Test,Post_Test,Unit_Test,Monthly_Test,...,Learning_Difficulty,Learning_Style,Participation_Score,Attention_Score,Confidence_Score,Collaboration_Score,Historical_Lessons,Average_Sequence_No,Completed_Historical_Lessons,Historical_Completion_Rate
0,ST001,78.00,1.75,11.25,4,100.0,50.0,77.5,74.0,73.5,...,Reading difficulty,Visual learner,3,2,3,3,4,2.5,2,50.0
1,ST002,78.75,1.50,11.25,4,100.0,47.5,65.0,65.0,68.0,...,Slow writing,Practical learner,1,1,2,2,4,2.5,2,50.0
2,ST003,92.25,1.00,16.00,4,100.0,60.0,82.5,80.0,81.0,...,None identified,Visual learner,3,3,3,3,4,2.5,2,50.0
3,ST004,68.75,2.00,11.75,4,75.0,37.5,57.5,58.0,61.0,...,Number recognition,Practical learner,2,2,2,3,4,2.5,1,25.0
4,ST005,95.25,1.00,16.00,4,100.0,60.0,85.0,85.0,88.5,...,None identified,Creative learner,3,3,3,4,4,2.5,3,75.0


In [103]:
# ============================================================
# STEP 20 - CHECK FEATURE MISSING VALUES
# ============================================================

print("Missing values in engineered features:")
print(student_features.isnull().sum())

print(
    "\nTotal missing values:",
    student_features.isnull().sum().sum()
)

Missing values in engineered features:
Student_ID                      0
Average_Score                   0
Average_Attempts                0
Average_Time_Taken              0
Total_Activities                0
Completion_Rate                 0
Pre_Test                        0
Post_Test                       0
Unit_Test                       0
Monthly_Test                    0
Term_Result                     0
Motor_Skills_Score              0
Creativity_Score                0
Interactive_Engagement          0
Test_Improvement                0
Participation                   0
Attention_Level                 0
Confidence                      0
Collaboration                   0
Learning_Difficulty             0
Learning_Style                  0
Participation_Score             0
Attention_Score                 0
Confidence_Score                0
Collaboration_Score             0
Historical_Lessons              0
Average_Sequence_No             0
Completed_Historical_Lessons    0
Historica

In [104]:
# ============================================================
# STEP 21 - CHECK FEATURE DATA TYPES
# ============================================================

print(student_features.dtypes)


Student_ID                       object
Average_Score                   float64
Average_Attempts                float64
Average_Time_Taken              float64
Total_Activities                  int64
Completion_Rate                 float64
Pre_Test                        float64
Post_Test                       float64
Unit_Test                       float64
Monthly_Test                    float64
Term_Result                     float64
Motor_Skills_Score              float64
Creativity_Score                float64
Interactive_Engagement          float64
Test_Improvement                float64
Participation                    object
Attention_Level                  object
Confidence                       object
Collaboration                    object
Learning_Difficulty              object
Learning_Style                   object
Participation_Score               int64
Attention_Score                   int64
Confidence_Score                  int64
Collaboration_Score               int64


In [105]:
# ============================================================
# STEP 22.1 - INSPECT ALL ENGINEERED FEATURES
# ============================================================

print("Number of features:", len(student_features.columns))

print("\nAll feature columns:")
for i, column in enumerate(student_features.columns, start=1):
    print(i, "->", column)

Number of features: 29

All feature columns:
1 -> Student_ID
2 -> Average_Score
3 -> Average_Attempts
4 -> Average_Time_Taken
5 -> Total_Activities
6 -> Completion_Rate
7 -> Pre_Test
8 -> Post_Test
9 -> Unit_Test
10 -> Monthly_Test
11 -> Term_Result
12 -> Motor_Skills_Score
13 -> Creativity_Score
14 -> Interactive_Engagement
15 -> Test_Improvement
16 -> Participation
17 -> Attention_Level
18 -> Confidence
19 -> Collaboration
20 -> Learning_Difficulty
21 -> Learning_Style
22 -> Participation_Score
23 -> Attention_Score
24 -> Confidence_Score
25 -> Collaboration_Score
26 -> Historical_Lessons
27 -> Average_Sequence_No
28 -> Completed_Historical_Lessons
29 -> Historical_Completion_Rate


In [106]:
# ============================================================
# STEP 22.2 - SAVE ENGINEERED FEATURES
# ============================================================

FEATURE_PATH = "../data/processed/student_features.csv"

student_features.to_csv(
    FEATURE_PATH,
    index=False
)

print("Student features saved successfully.")
print("Path:", FEATURE_PATH)

Student features saved successfully.
Path: ../data/processed/student_features.csv


In [107]:
import os

print(
    "File exists:",
    os.path.exists(FEATURE_PATH)
)

File exists: True


In [108]:
for i, column in enumerate(student_features.columns, start=1):
    print(i, "->", column)

1 -> Student_ID
2 -> Average_Score
3 -> Average_Attempts
4 -> Average_Time_Taken
5 -> Total_Activities
6 -> Completion_Rate
7 -> Pre_Test
8 -> Post_Test
9 -> Unit_Test
10 -> Monthly_Test
11 -> Term_Result
12 -> Motor_Skills_Score
13 -> Creativity_Score
14 -> Interactive_Engagement
15 -> Test_Improvement
16 -> Participation
17 -> Attention_Level
18 -> Confidence
19 -> Collaboration
20 -> Learning_Difficulty
21 -> Learning_Style
22 -> Participation_Score
23 -> Attention_Score
24 -> Confidence_Score
25 -> Collaboration_Score
26 -> Historical_Lessons
27 -> Average_Sequence_No
28 -> Completed_Historical_Lessons
29 -> Historical_Completion_Rate


In [109]:
student_features.dtypes

Student_ID                       object
Average_Score                   float64
Average_Attempts                float64
Average_Time_Taken              float64
Total_Activities                  int64
Completion_Rate                 float64
Pre_Test                        float64
Post_Test                       float64
Unit_Test                       float64
Monthly_Test                    float64
Term_Result                     float64
Motor_Skills_Score              float64
Creativity_Score                float64
Interactive_Engagement          float64
Test_Improvement                float64
Participation                    object
Attention_Level                  object
Confidence                       object
Collaboration                    object
Learning_Difficulty              object
Learning_Style                   object
Participation_Score               int64
Attention_Score                   int64
Confidence_Score                  int64
Collaboration_Score               int64


In [110]:

# ============================================================
# FINAL FEATURE ENGINEERING VALIDATION
# ============================================================

print("=" * 70)
print("FEATURE ENGINEERING COMPLETED")
print("=" * 70)

print("Dataset shape:", student_features.shape)
print("Number of students:", student_features.shape[0])
print("Number of features:", student_features.shape[1])

print("\nTotal missing values:",
      student_features.isnull().sum().sum())

print("\nDuplicate rows:",
      student_features.duplicated().sum())

print("\nFeature columns:")
print(student_features.columns.tolist())

print("=" * 70)

FEATURE ENGINEERING COMPLETED
Dataset shape: (15, 29)
Number of students: 15
Number of features: 29

Total missing values: 0

Duplicate rows: 0

Feature columns:
['Student_ID', 'Average_Score', 'Average_Attempts', 'Average_Time_Taken', 'Total_Activities', 'Completion_Rate', 'Pre_Test', 'Post_Test', 'Unit_Test', 'Monthly_Test', 'Term_Result', 'Motor_Skills_Score', 'Creativity_Score', 'Interactive_Engagement', 'Test_Improvement', 'Participation', 'Attention_Level', 'Confidence', 'Collaboration', 'Learning_Difficulty', 'Learning_Style', 'Participation_Score', 'Attention_Score', 'Confidence_Score', 'Collaboration_Score', 'Historical_Lessons', 'Average_Sequence_No', 'Completed_Historical_Lessons', 'Historical_Completion_Rate']


In [111]:
FEATURE_PATH = "../data/processed/student_features.csv"

student_features.to_csv(
    FEATURE_PATH,
    index=False
)

print("Feature dataset saved successfully.")
print("Saved to:", FEATURE_PATH)

Feature dataset saved successfully.
Saved to: ../data/processed/student_features.csv


In [112]:
print("Student features shape:", student_features.shape)

print("\nNumber of unique students:",
      student_features["Student_ID"].nunique())

print("\nStudent IDs:")
print(student_features["Student_ID"].tolist())

Student features shape: (15, 29)

Number of unique students: 15

Student IDs:
['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015']


In [113]:
print("Student features shape:", student_features.shape)

print("Number of unique students:",
      student_features["Student_ID"].nunique())

print("\nStudent IDs:")
print(sorted(student_features["Student_ID"].unique()))

Student features shape: (15, 29)
Number of unique students: 15

Student IDs:
['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015']


In [114]:
import pandas as pd

student_clean = pd.read_csv("../data/processed/student_clean.csv")

print("Student clean shape:", student_clean.shape)

print("\nNumber of unique students:",
      student_clean["Student_ID"].nunique())

print("\nStudent IDs:")
print(sorted(student_clean["Student_ID"].unique()))

Student clean shape: (100, 8)

Number of unique students: 100

Student IDs:
['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015', 'ST016', 'ST017', 'ST018', 'ST019', 'ST020', 'ST021', 'ST022', 'ST023', 'ST024', 'ST025', 'ST026', 'ST027', 'ST028', 'ST029', 'ST030', 'ST031', 'ST032', 'ST033', 'ST034', 'ST035', 'ST036', 'ST037', 'ST038', 'ST039', 'ST040', 'ST041', 'ST042', 'ST043', 'ST044', 'ST045', 'ST046', 'ST047', 'ST048', 'ST049', 'ST050', 'ST051', 'ST052', 'ST053', 'ST054', 'ST055', 'ST056', 'ST057', 'ST058', 'ST059', 'ST060', 'ST061', 'ST062', 'ST063', 'ST064', 'ST065', 'ST066', 'ST067', 'ST068', 'ST069', 'ST070', 'ST071', 'ST072', 'ST073', 'ST074', 'ST075', 'ST076', 'ST077', 'ST078', 'ST079', 'ST080', 'ST081', 'ST082', 'ST083', 'ST084', 'ST085', 'ST086', 'ST087', 'ST088', 'ST089', 'ST090', 'ST091', 'ST092', 'ST093', 'ST094', 'ST095', 'ST096', 'ST097', 'ST098', 'ST099', 'ST100']


In [115]:
import pandas as pd

files = {
    "student_clean": "../data/processed/student_clean.csv",
    "activity_clean": "../data/processed/activity_clean.csv",
    "assessment_clean": "../data/processed/assessment_clean.csv",
    "historical_clean": "../data/processed/historical_clean.csv",
    "ai_learning_clean": "../data/processed/ai_learning_clean.csv",
    "teacher_clean": "../data/processed/teacher_clean.csv"
}

for name, path in files.items():
    try:
        df = pd.read_csv(path)

        print("=" * 60)
        print(name)
        print("Rows:", len(df))

        if "Student_ID" in df.columns:
            print("Unique students:", df["Student_ID"].nunique())
            print("First IDs:", sorted(df["Student_ID"].unique())[:20])
        else:
            print("Student_ID column: NOT FOUND")

    except Exception as e:
        print(name, "ERROR:", e)

student_clean
Rows: 100
Unique students: 100
First IDs: ['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015', 'ST016', 'ST017', 'ST018', 'ST019', 'ST020']
activity_clean
Rows: 60
Unique students: 15
First IDs: ['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015']
assessment_clean
Rows: 30
Unique students: 15
First IDs: ['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015']
historical_clean
Rows: 60
Unique students: 15
First IDs: ['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', 'ST009', 'ST010', 'ST011', 'ST012', 'ST013', 'ST014', 'ST015']
ai_learning_clean
Rows: 11
Student_ID column: NOT FOUND
teacher_clean
Rows: 100
Unique students: 100
First IDs: ['ST001', 'ST002', 'ST003', 'ST004', 'ST005', 'ST006', 'ST007', 'ST008', '

In [116]:
activity_features = (
    activity_clean
    .groupby("Student_ID")
    .agg(
        Activity_Count=("Activity_ID", "count"),
        Average_Activity_Score=("Score", "mean"),
        Average_Time_Taken=("Time_Taken_Min", "mean"),
        Average_Attempts=("Attempts", "mean")
    )
    .reset_index()
)

In [117]:
print("Activity students:",
      activity_features["Student_ID"].nunique())

Activity students: 15


In [118]:
# ============================================================
# FINAL CHECK — VERIFY SAVED FEATURE DATASET
# ============================================================

print("=" * 70)
print("FINAL FEATURE DATASET CHECK")
print("=" * 70)


saved_features = pd.read_csv(
    FEATURE_PATH
)


print(
    "Dataset shape:",
    saved_features.shape
)


print(
    "Number of students:",
    saved_features["Student_ID"].nunique()
)


print(
    "Number of features:",
    saved_features.shape[1]
)


print(
    "Missing values:",
    saved_features.isnull().sum().sum()
)


print(
    "Duplicate rows:",
    saved_features.duplicated().sum()
)


print("\n✓ Saved feature dataset verified.")

FINAL FEATURE DATASET CHECK
Dataset shape: (15, 29)
Number of students: 15
Number of features: 29
Missing values: 0
Duplicate rows: 0

✓ Saved feature dataset verified.
